<a href="https://colab.research.google.com/github/heberdavi/mba-engsoft-tcc/blob/main/notebooks/02-visualizacao_resultados.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Célula 1: Configuração e Criação da Pasta de Exportação
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
from google.colab import drive
import os

# 1. Montagem do Drive
drive.mount('/content/drive')

# 2. Definição de Caminhos (AJUSTE O CAMINHO DO SEU BANCO ABAIXO)
DB_PATH = '/content/drive/My Drive/mba-engsof-tcc/v5/base-dados-v5.db'
EXPORT_PATH = '/content/drive/My Drive/mba-engsof-tcc/v5/graficos-tcc'

# Cria a pasta de exportação se ela não existir
if not os.path.exists(EXPORT_PATH):
    os.makedirs(EXPORT_PATH)
    print(f"📂 Pasta criada: {EXPORT_PATH}")

def get_connection():
    return sqlite3.connect(DB_PATH)

# Garante suporte a acentuação e visual limpo
sns.set_context("paper", font_scale=1.2)

# Configurações para qualidade ds imagens
sns.set_theme(style="whitegrid")
plt.rcParams['figure.dpi'] = 300
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12

print("✅ Ambiente configurado para exportação de imagens JPG.")

In [ ]:
# Célula 2: Geração de Nuvens de Palavras (Apenas Versos Positivos com Diferenciação por Autor)
import nltk
from nltk.corpus import stopwords
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
from wordcloud import WordCloud
import os

nltk.download('stopwords')

def executar_nuvens_antidotos_positivos_refinada():
    conn = get_connection()

    # 1. Seleciona os eixos (excluindo o narrativo)
    query_antidotos = "SELECT id, antidoto_referencia FROM topico WHERE id != 3"
    df_topicos = pd.read_sql_query(query_antidotos, conn)

    # 2. Stopwords: Limpeza de resíduos geográficos e estruturais bíblicos
    stop_words_pt = set(stopwords.words('portuguese'))
    custom_stops = {
        'disse', 'então', 'veio', 'porque', 'pois', 'sobre', 'todos', 'tudo',
        'assim', 'ainda', 'outra', 'outros', 'será', 'pode', 'fazer', 'tão',
        'casa', 'filho', 'filhos', 'homem', 'mulher', 'terra', 'povo', 'rei',
        'senhor', 'deus', 'jesus', 'cristo', 'amém', 'ora', 'eis', 'vós',
        'teu', 'tua', 'meu', 'minha', 'toda', 'ano', 'anos', 'morreu', 'mortos',
        # Adicionando ruídos geográficos/históricos comuns que "sujam" a análise existencial
        'israel', 'judá', 'jerusalém', 'egito', 'babilônia', 'filisteus', 'moisés', 'davi'
    }
    todas_stops = stop_words_pt.union(custom_stops)

    # 3. Dicionário de Cores para Identidade Visual do TCC
    # Han = Tons de Verde/Ciano (Descanso/Vigor)
    # Bauman = Tons de Outono/Laranja (Solidez/Rocha)
    # Frankl = Tons de Azul/Roxo (Profundidade/Sentido)
    cores_eixos = {
        0: 'viridis',      # Esgotamento (Han)
        1: 'YlOrBr',       # Transitoriedade (Bauman)
        2: 'coolwarm'      # Insignificância (Frankl)
    }

    print("🌟 Gerando Nuvens de Antídotos (Filtro: Sentimento Positivo)...")

    for _, row in df_topicos.iterrows():
        t_id = row['id']
        nome_antidoto = row['antidoto_referencia']

        # Consulta com filtro de sentimento POSITIVO
        query_texto = f"""
            SELECT vl.texto_limpo
            FROM verso_limpo vl
            JOIN verso_topico vt ON vl.verso_id = vt.verso_id
            JOIN verso_sentimento s ON vl.verso_id = s.verso_id
            WHERE vt.topico_id = {t_id}
            AND s.sentimento_num = 1
        """
        df_textos = pd.read_sql_query(query_texto, conn)

        if not df_textos.empty:
            texto_final = " ".join(df_textos['texto_limpo'].fillna('').tolist())

            # Configuração estética focada em legibilidade acadêmica
            wordcloud = WordCloud(width=1600, height=900,
                                  background_color='white',
                                  max_words=60, # Reduzido para destacar apenas o essencial
                                  stopwords=todas_stops,
                                  colormap=cores_eixos.get(t_id, 'plasma'),
                                  collocations=False,
                                  prefer_horizontal=0.85).generate(texto_final)

            plt.figure(figsize=(14, 8))
            plt.imshow(wordcloud, interpolation='bilinear')
            plt.axis('off')

            # Título interno opcional (comentado caso queira o gráfico limpo para o Word)
            # plt.title(f"Antídotos: {nome_antidoto}", fontsize=18, pad=20)

            # Nome de arquivo limpo e padronizado
            safe_name = nome_antidoto.split('(')[0].strip().lower().replace(' ', '_')
            file_name = f"nuvem_cura_{safe_name}.jpg"

            plt.savefig(os.path.join(EXPORT_PATH, file_name), dpi=300, bbox_inches='tight')
            plt.close()
            print(f"✅ Nuvem de 'Cura' salva: {file_name} (Esquema de cores: {cores_eixos.get(t_id)})")
        else:
            print(f"⚠️ Versos positivos insuficientes para: {nome_antidoto}")

    conn.close()

executar_nuvens_antidotos_positivos_refinada()

In [ ]:
# Célula 3: Distribuição de ANTÍDOTOS (Positivos) por Gênero Literário
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

def gerar_grafico_antidotos_por_genero_filtrado():
    conn = get_connection()

    # 1. Consulta SQL Refinada: Apenas Versos POSITIVOS (Antídotos)
    query = """
        SELECT
            g.nome as Genero,
            t.antidoto_referencia as Antidoto,
            COUNT(vt.verso_id) as Frequencia
        FROM verso_topico vt
        JOIN topico t ON vt.topico_id = t.id
        JOIN verso_sentimento vs ON vt.verso_id = vs.verso_id
        JOIN verso v ON vt.verso_id = v.id
        JOIN livro l ON v.livro_id = l.id
        JOIN genero_literario g ON l.genero_id = g.id
        WHERE vs.sentimento_num = 1  -- FILTRO CRUCIAL: Apenas a Cura
          AND t.id != 3              -- Exclui Narrativo/Outros
        GROUP BY Genero, Antidoto
    """

    df_dist = pd.read_sql_query(query, conn)
    conn.close()

    if df_dist.empty:
        print("⚠️ Dados não encontrados. Verifique se a Célula 6 (Sentimentos) foi executada.")
        return

    # 2. Pivotar e Ordenar
    df_pivot = df_dist.pivot(index='Genero', columns='Antidoto', values='Frequencia').fillna(0)

    # Ordenar pelo total de antídotos para um visual mais organizado (escadinha)
    df_pivot['Total'] = df_pivot.sum(axis=1)
    df_pivot = df_pivot.sort_values(by='Total', ascending=True).drop(columns='Total')

    # 3. Configuração Estética
    sns.set_style("white")

    # Paleta de cores harmonizada com as Nuvens de Palavras
    # Azul (Frankl), Verde (Han), Laranja (Bauman)
    cores_harmonizadas = ['#2E8B57', '#4682B4', '#D2691E']

    ax = df_pivot.plot(kind='barh',
                       stacked=True,
                       color=cores_harmonizadas,
                       figsize=(14, 8),
                       width=0.75,
                       edgecolor='white',
                       linewidth=1)

    # Rótulos
    plt.xlabel('Volume de Antídotos (Versículos Positivos)', fontsize=12, labelpad=15)
    plt.ylabel('Gênero Literário', fontsize=12, labelpad=15)

    # Adicionando os valores numéricos dentro ou fora das barras para facilitar a leitura da banca
    for p in ax.patches:
        width = p.get_width()
        if width > 5: # Só desenha se a barra for visível
            ax.annotate(f'{int(width)}',
                        (p.get_x() + width / 2, p.get_y() + p.get_height() / 2),
                        ha='center', va='center',
                        fontsize=9, color='white', fontweight='bold')

    sns.despine(left=False, bottom=False)
    plt.legend(title='Eixos Existenciais', bbox_to_anchor=(1.02, 1), loc='upper left', frameon=False)
    plt.tight_layout()

    # 4. Exportação
    file_name = "distribuicao_antidotos_genero_final.jpg"
    plt.savefig(os.path.join(EXPORT_PATH, file_name), dpi=300, bbox_inches='tight')

    print(f"✅ Gráfico de Antídotos salvo com sucesso: {file_name}")
    plt.show()

gerar_grafico_antidotos_por_genero_filtrado()

In [ ]:
# Célula 4: Gauge Bipolar
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import os

# Desativa avisos de plotagem em lote (para um log mais limpo)
plt.rcParams.update({'figure.max_open_warning': 0})

def criar_gauge_ultra_minimal(valor, nome_arquivo, cor_eixo):
    # Configuração da Escala Bipolar: de -0.5 (Crise) a +0.5 (Cura)
    limite = 0.5

    # Mapeamento do valor para o semicírculo de 180 graus
    # -0.5 -> 0° | 0.0 -> 90° (Centro) | +0.5 -> 180°
    posicao_graus = ((valor + limite) / (2 * limite)) * 180
    posicao_graus = max(0, min(180, posicao_graus)) # Clamp de segurança

    fig, ax = plt.subplots(figsize=(6, 3)) # Proporção 2:1 para semicírculo

    # 1. Arco de Fundo (Fundo Neutro Cinza Claro)
    ax.pie([180, 180], colors=['#F2F2F2', 'white'], startangle=180,
           wedgeprops={'width': 0.35, 'edgecolor': 'white'})

    # 2. Ponteiro Indicador
    # Desenhamos uma pequena fatia (3 graus de largura para visibilidade)
    ax.pie([posicao_graus - 1.5, 3, 180 - posicao_graus - 1.5, 180],
           colors=['#F2F2F2', cor_eixo, '#F2F2F2', 'white'],
           startangle=180, counterclock=True,
           wedgeprops={'width': 0.4, 'edgecolor': 'none'})

    # 3. Exibição Pura do Valor Central
    # Aumentamos o tamanho e centralizamos o valor numérico
    plt.text(0, 0.1, f"{valor:.3f}", ha='center', va='center',
             fontsize=36, fontweight='bold', color='#333333')

    # 4. Remoção Total de Elementos de Texto e Eixos
    # Removemos os marcadores 'CRISE', 'CURA', 'NEUTRO' e o nome do autor
    ax.axis('equal')
    ax.set_xticks([])
    ax.set_yticks([])
    sns.despine(left=True, bottom=True) # Remove as bordas do Seaborn

    # Salvamento Automatizado com Padding Mínimo
    # dpi=300 garante qualidade de impressão na tese
    path_completo = os.path.join(EXPORT_PATH, nome_arquivo)
    plt.savefig(path_completo, dpi=300, bbox_inches='tight', pad_inches=0.01)
    plt.close(fig) # Fecha explicitamente para economizar memória do Colab
    print(f"✅ Gauge Ultra-Minimalista salvo: {nome_arquivo} (Valor real: {valor:.4f})")

def processar_todos_os_gauges_minimalistas():
    # Sincroniza com o caminho definido na Célula 1
    # Certifique-se de que EXPORT_PATH e get_connection estão definidos
    try:
        conn = get_connection()

        # CONSULTA DINÂMICA: Busca a média de sentimento por eixo existencial
        query = """
            SELECT t.id, t.antidoto_referencia, AVG(vs.sentimento_num) as media_polaridade
            FROM verso_topico vt
            JOIN topico t ON vt.topico_id = t.id
            JOIN verso_sentimento vs ON vt.verso_id = vs.verso_id
            WHERE t.id IN (0, 1, 2)
            GROUP BY t.id, t.antidoto_referencia
        """
        df_resultados = pd.read_sql_query(query, conn)
        conn.close()
    except Exception as e:
        print(f"❌ Erro na consulta do banco: {e}")
        return

    if df_resultados.empty:
        print("❌ Erro: Nenhum dado de sentimento encontrado. Execute a Célula 6 primeiro.")
        return

    print(f"🚀 Iniciando geração de {len(df_resultados)} Gauges Ultra-Minimalistas...")

    # Cores temáticas para alinhar com as Nuvens e o Stack Chart
    # Isso demonstra cuidado UX na sua tese
    # Han = Verde, Bauman = Laranja, Frankl = Azul
    cores_eixos = {0: '#2E8B57', 1: '#D2691E', 2: '#4682B4'}

    for _, row in df_resultados.iterrows():
        # Gera nome de arquivo limpo
        t_id = row['id']
        safe_name = row['antidoto_referencia'].split('(')[0].strip().lower().replace(' ', '_')
        file_name = f"gauge_puro_minimal_{safe_name}.jpg"

        # Usa a cor temática do autor
        cor_ativa = cores_eixos.get(t_id, '#50C878') # Padrão verde se falhar

        # Executa o desenho do gráfico limpo
        criar_gauge_ultra_minimal(
            valor=row['media_polaridade'],
            nome_arquivo=file_name,
            cor_eixo=cor_ativa
        )

# Execução principal
processar_todos_os_gauges_minimalistas()

In [ ]:
# Célula 5: Comparativo Crise vs. Cura por Gênero Literário (Versão Corrigida)
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

def gerar_grafico_crise_vs_cura_robusto():
    conn = get_connection()

    # 1. Consulta SQL: Focada em garantir que todos os gêneros com dados apareçam
    # Usamos JOINs explícitos para garantir a integridade entre verso_id e verso_id
    query = """
        SELECT
            g.nome as Genero,
            SUM(CASE WHEN vs.sentimento_num = 1 THEN 1 ELSE 0 END) as Cura_Positivo,
            SUM(CASE WHEN vs.sentimento_num = -1 THEN 1 ELSE 0 END) as Crise_Negativo
        FROM genero_literario g
        JOIN livro l ON g.id = l.genero_id
        JOIN verso v ON l.id = v.livro_id
        JOIN verso_topico vt ON v.id = vt.verso_id
        JOIN verso_sentimento vs ON v.id = vs.verso_id
        WHERE vt.topico_id IN (0, 1, 2)
        GROUP BY g.nome
        HAVING (Cura_Positivo + Crise_Negativo) > 0
        ORDER BY (Cura_Positivo + Crise_Negativo) DESC
    """

    df_comp = pd.read_sql_query(query, conn)
    conn.close()

    if df_comp.empty:
        print("⚠️ A consulta não retornou dados. Verifique se as tabelas vt e vs estão povoadas.")
        return

    # 2. Preparação dos Dados (Melt)
    df_melted = df_comp.melt(id_vars='Genero', var_name='Tipo', value_name='Quantidade')
    df_melted['Tipo'] = df_melted['Tipo'].replace({
        'Cura_Positivo': 'Antídoto (Cura)',
        'Crise_Negativo': 'Problemática (Crise)'
    })

    # 3. Gráfico
    plt.figure(figsize=(14, 8))
    sns.set_style("whitegrid")

    # Cores acadêmicas: Verde para Cura, Coral para Crise
    paleta = {'Antídoto (Cura)': '#50C878', 'Problemática (Crise)': '#FF6B6B'}

    ax = sns.barplot(data=df_melted, x='Quantidade', y='Genero', hue='Tipo',
                     palette=paleta, edgecolor='white')

    # Ajustes estéticos
    plt.xlabel('Volume de Versículos (Frequência Absoluta)', fontsize=11)
    plt.ylabel('Gênero Literário', fontsize=11)
    plt.legend(title='Análise de Sentimento', loc='lower right', frameon=True)

    # Rótulos nas barras
    for p in ax.patches:
        val = int(p.get_width())
        if val > 0:
            ax.annotate(f'{val}',
                        (p.get_width(), p.get_y() + p.get_height() / 2),
                        ha='left', va='center', fontsize=9, xytext=(5, 0),
                        textcoords='offset points')

    sns.despine()
    plt.tight_layout()

    # 4. Salvamento
    file_name = "comparativo_crise_cura_final.jpg"
    plt.savefig(os.path.join(EXPORT_PATH, file_name), dpi=300, bbox_inches='tight')
    plt.show()
    print(f"✅ Gráfico salvo com sucesso: {file_name}")

gerar_grafico_crise_vs_cura_robusto()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import os

def gerar_workflow_data_prep_destaque():
    # Proporção ampla para garantir que as fontes maximizadas respirem
    fig, ax = plt.subplots(figsize=(24, 7), facecolor='white')
    ax.set_facecolor('white')

    etapas = [
        {"n": "1", "titulo": "Ingestão", "sub": "Business Und.", "libs": "google.colab\nos, gc", "tipo": "und"},
        {"n": "2", "titulo": "Ambiente", "sub": "Data Prep", "libs": "transformers\ntorch", "tipo": "prep"},
        {"n": "3", "titulo": "Carga", "sub": "Data Prep", "libs": "sqlite3\npandas", "tipo": "prep"},
        {"n": "4", "titulo": "Limpeza", "sub": "Data Prep", "libs": "re\nstring", "tipo": "prep"},
        {"n": "5", "titulo": "Tópicos", "sub": "Modeling", "libs": "bertopic\nsklearn, nltk", "tipo": "mod"},
        {"n": "6", "titulo": "Sentimento", "sub": "Modeling", "libs": "pysentimiento\ntqdm", "tipo": "mod"},
        {"n": "7", "titulo": "Avaliação", "sub": "Evaluation", "libs": "matplotlib\nseaborn, wordcloud", "tipo": "eval"}
    ]

    # Esquema de cores refinado para diferenciar Data Prep
    cores = {
        "und":  {"face": "#F5F5F5", "edge": "#9E9E9E", "text": "#424242", "lib_color": "#616161"}, # Cinza
        "prep": {"face": "#FFF3E0", "edge": "#FF9800", "text": "#E65100", "lib_color": "#EF6C00"}, # Laranja (Data Prep)
        "mod":  {"face": "#E3F2FD", "edge": "#1976D2", "text": "#0D47A1", "lib_color": "#1565C0"}, # Azul
        "eval": {"face": "#E8F5E9", "edge": "#388E3C", "text": "#1B5E20", "lib_color": "#2E7D32"}  # Verde
    }

    n_etapas = len(etapas)
    box_w, box_h = 1.25, 0.95
    espacamento = 1.65

    # Linha conectora de fundo
    ax.plot([0, (n_etapas-1) * espacamento], [0.5, 0.5], color='#F0F0F0',
            linewidth=15, zorder=1, solid_capstyle='round')

    for i, etapa in enumerate(etapas):
        x = i * espacamento
        y = 0.5
        estilo = cores[etapa["tipo"]]

        # 1. Box da Etapa
        rect = patches.FancyBboxPatch(
            (x - box_w/2, y - box_h/2), box_w, box_h,
            boxstyle="round,pad=0.04", linewidth=2.8,
            edgecolor=estilo["edge"], facecolor=estilo["face"], zorder=3
        )
        ax.add_patch(rect)

        # 2. Rótulo Etapa X
        ax.text(x - box_w/2, y + box_h/2 + 0.08, f"Etapa {etapa['n']}",
                fontsize=13, fontweight='bold', color='#757575', ha='left')

        # 3. Título Principal (Max)
        ax.text(x, y + 0.25, etapa["titulo"], ha='center', va='center',
                fontsize=18, fontweight='black', color=estilo["text"], zorder=4)

        # 4. Subtítulo CRISP-DM
        ax.text(x, y + 0.08, etapa["sub"], ha='center', va='center',
                fontsize=12, style='italic', color=estilo["text"], alpha=0.9, zorder=4)

        # 5. Bibliotecas Monospace
        ax.text(x, y - 0.22, etapa["libs"], ha='center', va='center',
                fontsize=12, fontweight='bold', color=estilo["lib_color"],
                family='monospace', zorder=4)

        # 6. Setas (Cores seguem a origem do fluxo)
        if i < n_etapas - 1:
            ax.annotate("", xy=(x + espacamento - box_w/2 - 0.06, y),
                        xytext=(x + box_w/2 + 0.06, y),
                        arrowprops=dict(arrowstyle='-|>', color=estilo["edge"],
                        lw=2.5, mutation_scale=25), zorder=2)

    ax.set_xlim(-1.0, (n_etapas - 1) * espacamento + 1.0)
    ax.set_ylim(-0.1, 1.1)
    ax.axis('off')

    plt.tight_layout()

    file_name = "workflow_crisp_data_prep_final.jpg"
    plt.savefig(os.path.join(EXPORT_PATH, file_name), dpi=300, bbox_inches='tight', pad_inches=0.1)

    print(f"✅ Workflow com Data Prep destacado gerado: {file_name}")
    plt.show()

gerar_workflow_data_prep_destaque()